In [2]:
import torch
import numpy as np
import pandas as pd

In [ ]:
#Read the original data in my workspace

In [6]:
RAW_PATH = r"C:\Users\guest1\OneDrive\Desktop\data analytics portfolio projects\Telecom Churn Analysis\data\ember_mobile_raw.csv"

df = pd.read_csv(
    RAW_PATH,
    parse_dates=[
        "signup_date", "contract_start_date", "contract_end_date",
        "disconnection_date", "last_payment_date", "last_activity_date"
    ],
    dayfirst=True,   # your dates are DD/MM/YYYY
)

print(f"Shape: {df.shape}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nHead:\n{df.head()}")
print(f"\nChurn breakdown:\n{df['account_status'].value_counts()}")
print(f"\nMissing values (top 10):\n{df.isna().sum().sort_values(ascending=False).head(10)}")

Shape: (21346, 28)

Dtypes:
customerID                         str
signup_date             datetime64[us]
account_status                     str
contract_start_date     datetime64[us]
contract_end_date       datetime64[us]
disconnection_date      datetime64[us]
disconnection_reason               str
last_payment_date       datetime64[us]
last_activity_date      datetime64[us]
gender                             str
SeniorCitizen                    int64
Partner                            str
Dependents                         str
tenure                           int64
PhoneService                       str
MultipleLines                      str
InternetService                    str
OnlineSecurity                     str
OnlineBackup                       str
DeviceProtection                   str
TechSupport                        str
StreamingTV                        str
StreamingMovies                    str
Contract                           str
PaperlessBilling                   s

#Using a hugging face trained Transformer model for sentiment text analysis

In [7]:

from transformers import pipeline

RAW_PATH = r"C:\Users\guest1\OneDrive\Desktop\data analytics portfolio projects\Telecom Churn Analysis\data\ember_mobile_raw.csv"
OUT_PATH = r"C:\Users\guest1\OneDrive\Desktop\data analytics portfolio projects\Telecom Churn Analysis\data\ember_mobile_sentiment.csv"


df = pd.read_csv(RAW_PATH, dayfirst=True)

#  Category mapping
category_map = {
    "Poor customer service": "Service Quality",
    "Poor coverage":         "Service Quality",
    "Price too high":        "Pricing",
    "Competitor offer":      "Pricing",
    "Switched provider":     "Pricing",
    "No longer needed":      "Life Change",
    "Moved to family plan":  "Life Change",
    "Fraud":                 "Account Issue",
    "Contract violation":    "Account Issue",
    "Non-payment":           "Account Issue",
}
df["comment_category"] = df["disconnection_reason"].map(category_map)

#Sentiment — only on rows that have text
mask = df["disconnection_reason"].notna()
texts = df.loc[mask, "disconnection_reason"].astype(str).tolist()

clf = pipeline(
    "sentiment-analysis",
    model="cardiffnlp/twitter-roberta-base-sentiment-latest",  # 3-class: neg/neu/pos
    device=-1,   #No CUDA device available, so use CPU
    truncation=True,
    max_length=128,
)
results = clf(texts, batch_size=64)

df.loc[mask, "sentiment_label"] = [r["label"] for r in results]
df.loc[mask, "sentiment_score"] = [r["score"] for r in results]


df.to_csv(OUT_PATH, index=False)
print(f"Saved {OUT_PATH}")
print(df[["disconnection_reason", "comment_category", "sentiment_label", "sentiment_score"]]
      .dropna().head(10))
print("\nSentiment breakdown:")
print(df["sentiment_label"].value_counts())
print("\nCategory breakdown:")
print(df["comment_category"].value_counts())

config.json:   0%|          | 0.00/929 [00:00<?, ?B/s]

c:\Users\guest1\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:141: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\guest1\.cache\huggingface\hub\models--cardiffnlp--twitter-roberta-base-sentiment-latest. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


pytorch_model.bin: reconstructing file:   0%|          |  0.00B /  501MB            

pytorch_model.bin: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

...

model.safetensors: reconstructing file:   0%|          |  0.00B /  501MB            

model.safetensors: downloading bytes:           |  0.00B            

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

Saved C:\Users\guest1\OneDrive\Desktop\data analytics portfolio projects\Telecom Churn Analysis\data\ember_mobile_sentiment.csv
     disconnection_reason comment_category sentiment_label  sentiment_score
9          Price too high          Pricing        negative         0.624388
11       No longer needed      Life Change         neutral         0.642227
14  Poor customer service  Service Quality        negative         0.844663
21       No longer needed      Life Change         neutral         0.642227
24       No longer needed      Life Change         neutral         0.642227
31       No longer needed      Life Change         neutral         0.642227
33  Poor customer service  Service Quality        negative         0.844663
36       No longer needed      Life Change         neutral         0.642227
37         Price too high          Pricing        negative         0.624388
41       Competitor offer          Pricing         neutral         0.780465

Sentiment breakdown:
sentiment_labe